In [ ]:
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA

import nltk

nltk.download('punkt')
nltk.download('stopwords')

In [ ]:
with open("foodie.txt", "r", encoding="utf-8") as file:
    original_text = file.read()

print("Original Text:\n")
print(original_text)

In [ ]:
docs = sent_tokenize(original_text)

print("Documents:\n")

for i, document in enumerate(docs, 1):
    print(f"D{i}: {document}")

print("\nTotal Number of Documents:", len(docs))

In [ ]:
query = "favorite spicy biryani and pizza"

print("Query:\n")
print(query)

stop_words = set(stopwords.words("english"))

def preprocess_text(text):

    text = text.lower()

    text = text.translate(
        str.maketrans("", "", string.punctuation)
    )

    words = word_tokenize(text)

    filtered_words = [
        word
        for word in words
        if word not in stop_words
    ]

    return " ".join(filtered_words)

In [ ]:
processed_docs = [
    preprocess_text(doc)
    for doc in docs
]

processed_query = preprocess_text(query)

print("Processed Documents:\n")

for i, doc in enumerate(processed_docs, 1):
    print(f"D{i}: {doc}")

print("\nProcessed Query:")
print(processed_query)

In [ ]:
all_texts = processed_docs + [processed_query]

print("Documents:", len(processed_docs))
print("Total Texts:", len(all_texts))

In [ ]:
labels = [f"D{i}" for i in range(1, len(docs)+1)]

labels.append("Query")

print(labels)

In [ ]:
vectorizer = TfidfVectorizer()

tfidf_matrix = vectorizer.fit_transform(all_texts)

tfidf_array = tfidf_matrix.toarray()

feature_names = vectorizer.get_feature_names_out()

print("Shape:", tfidf_array.shape)
print("Vocabulary Size:", len(feature_names))

In [ ]:
tfidf_table = pd.DataFrame(
    tfidf_array,
    index=labels,
    columns=feature_names
)

display(tfidf_table.round(3))

In [ ]:
for i, label in enumerate(labels):

    print(f"\n{label} TF-IDF Scores:\n")

    for word, score in zip(feature_names, tfidf_array[i]):

        if score > 0:
            print(f"{word}: {score:.3f}")

In [ ]:
doc_vectors = tfidf_array[:-1]

query_vector = tfidf_array[-1].reshape(1,-1)

cos_sim = cosine_similarity(
    doc_vectors,
    query_vector
).flatten()

print("Cosine Similarities:\n")

for i, score in enumerate(cos_sim,1):
    print(f"D{i}: {score:.3f}")

In [ ]:
similarity_table = pd.DataFrame({

    "Document": labels[:-1],

    "Sentence": docs,

    "Cosine Similarity": cos_sim

})

display(similarity_table.round(3))

In [ ]:
sorted_similarity_table = similarity_table.sort_values(
    by="Cosine Similarity",
    ascending=False
).reset_index(drop=True)

display(sorted_similarity_table.round(3))

In [ ]:
best = np.argmax(cos_sim)

print("Most Relevant Sentence:\n")

print(docs[best])

print("\nSimilarity Score:", round(cos_sim[best],3))

In [ ]:
pca = PCA(n_components=2)

reduced = pca.fit_transform(tfidf_array)

pca_table = pd.DataFrame(
    reduced,
    index=labels,
    columns=["PC1","PC2"]
)

display(pca_table.round(3))

In [ ]:
plt.figure(figsize=(8,6))

for i, label in enumerate(labels):

    plt.scatter(
        reduced[i,0],
        reduced[i,1]
    )

    plt.text(
        reduced[i,0]+0.01,
        reduced[i,1]+0.01,
        label
    )

plt.title("TF-IDF Document Similarity using PCA")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.grid(True)

plt.show()